In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB2")

os.environ["WANDB_API_KEY"] = secret_value_0  # force it into the env so the SDK can see it
os.environ['MPLBACKEND'] = 'agg'  # Hoặc del os.environ['MPLBACKEND'] nếu không cần backend cụ thể
os.environ['ENABLE_PJRT_COMPATIBILITY'] = '1' # tpu v5e mới quá dùng jax hơi cũ nên phải setup
os.environ['JAX_TRACEBACK_FILTERING'] = 'off'

In [ ]:
!pip install -q tfds apache_beam mlcroissant
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] += ":/root/.local/bin"

In [ ]:
!cp -r /kaggle/input/shortcut-celebahq256/tensorflow_datasets /root
%cd /kaggle/working
!git clone https://github.com/kvfrans/tfds_builders.git
%cd tfds_builders/celebahq256
!tfds build

In [ ]:

%cd /kaggle/working
!git clone https://github.com/Bangchis/shortcut-models
%cd shortcut-models
branch = "gmm"
!git fetch --all
!git checkout {branch}
!git pull
!uv sync 1>sync_out.txt 2>sync_err.txt
!cp -r /kaggle/input/shortcut-celebahq256/data /kaggle/working/shortcut-models/


In [ ]:
%%bash -s "$secret_value_0"
cat > ~/.netrc <<EOF
machine api.wandb.ai login $1
EOF
chmod 600 ~/.netrc

In [ ]:

%cd /kaggle/working/shortcut-models
!mkdir -p /kaggle/working/gmm_diagnostics
!uv run data_prep.py \
--dataset_name celebahq256 \
--tfds_data_dir /kaggle/input/shortcut-celebahq256/tensorflow_datasets \
--batch_size 64 \
--gmm_save_path /kaggle/working/celebahq256_gmm_stats.npz \
--gmm_latent_cache_path /kaggle/working/celebahq256_gmm_latents.dat \
--gmm_num_modes 64 \
--gmm_fit_samples 32768 \
--gmm_valid_samples 4096 \
--gmm_em_iters 25 \
--gmm_em_restarts 1 \
--gmm_init_seed 0 \
--gmm_standardize_eps 1e-6 \
--gmm_pi_prior_strength 1e-2 \
--gmm_min_std 0.0 \
--gmm_min_std_data_frac 1.0 \
--gmm_kmeanspp_init 1 \
--gmm_em_chunk_size 128 \
--gmm_keep_latent_cache 0 \
--metrics_output_path /kaggle/working/gmm_diagnostics/gmm_metrics.json \
--wandb.name "prepare_gmm" \
> /kaggle/working/gmm_diagnostics/gmm_prep_stdout.txt \
2> /kaggle/working/gmm_diagnostics/gmm_prep_stderr.txt


In [ ]:

%cd /kaggle/working/shortcut-models
!git fetch --all
!git checkout {branch}
!git pull
!mkdir -p /kaggle/working/ckpts/celebahq256_gmm /kaggle/working/gmm_diagnostics
!uv run train.py \
--model.hidden_size 768 \
--model.patch_size 2 \
--model.depth 12 \
--model.num_heads 12 \
--model.mlp_ratio 4 \
--model.train_type naive \
--model.cfg_scale 0 \
--model.class_dropout_prob 1 \
--model.num_classes 1 \
--model.denoise_timesteps 128 \
--batch_size 64 \
--dataset_name celebahq256 \
--tfds_data_dir /kaggle/input/shortcut-celebahq256/tensorflow_datasets \
--fid_stats data/celeba256_fidstats_ours.npz \
--max_steps 500000 \
--eval_interval 50000 \
--log_interval 100 \
--save_dir /kaggle/working/ckpts/celebahq256_gmm \
--wandb.name gmm \
--model.weight_decay 0.01 \
--model.gmm_stats_path /kaggle/working/celebahq256_gmm_stats.npz \
--model.gmm_cond_channels 64 \
--eval_fid_timesteps 1,4,32,128 \
--metrics_output_path /kaggle/working/gmm_diagnostics/train_metrics.jsonl \
> /kaggle/working/gmm_diagnostics/train_stdout.txt \
2> /kaggle/working/gmm_diagnostics/train_stderr.txt
